# Project 1: reproducible baseline

This notebook uses only NumPy and the functions provided in the project. The ID column is kept only for the final submission and is never used as a feature.

In [6]:
import numpy as np
from helpers import load_csv_data, create_csv_submission
from implementations import logistic_regression

np.random.seed(42)
data = "data/dataset"
x_train, x_test, y_train, ids_train, ids_test = load_csv_data(data)
print(x_train.shape, x_test.shape, y_train.shape)
print("class proportions:", np.mean(y_train == -1), np.mean(y_train == 1))

(328135, 321) (109379, 321) (328135,)
class proportions: 0.911697929205967 0.08830207079403295


## 1. Diagnostics and preprocessing

Missing values are loaded as `NaN`. We remove constant columns and impute every remaining column with a median computed on the training data.

In [7]:
print("NaNs in train/test:", np.isnan(x_train).sum(), np.isnan(x_test).sum())
print("constant columns:", np.sum(np.nanmax(x_train, axis=0) == np.nanmin(x_train, axis=0)))

NaNs in train/test: 47175779 15724379
constant columns: 6


In [8]:
def fit_preprocessor(x):
    constant = np.nanmax(x, axis=0) == np.nanmin(x, axis=0)
    medians = np.nanmedian(x, axis=0)
    medians = np.where(np.isfinite(medians), medians, 0.0)
    return constant, medians

def transform(x, constant, medians, means=None, scales=None):
    z = x[:, ~constant].copy()
    med = medians[~constant]
    missing = np.isnan(z)
    z[missing] = np.take(med, np.where(missing)[1])
    if means is None:
        means = np.mean(z, axis=0)
        scales = np.std(z, axis=0)
        scales = np.where(scales < 1e-12, 1.0, scales)
    return (z - means) / scales, means, scales

constant, medians = fit_preprocessor(x_train)
x_train_clean, means, scales = transform(x_train, constant, medians)
x_test_clean, _, _ = transform(x_test, constant, medians, means, scales)
tx_train = np.c_[np.ones(x_train_clean.shape[0]), x_train_clean]
tx_test = np.c_[np.ones(x_test_clean.shape[0]), x_test_clean]
y01 = (y_train == 1).astype(float)
print("processed shapes:", tx_train.shape, tx_test.shape)

processed shapes: (328135, 316) (109379, 316)


## 2. Fixed stratified validation split

Use this split to compare baselines and hyperparameters. For the report, repeat this with several folds and record an ablation table.

In [9]:
positive = np.flatnonzero(y01 == 1)
negative = np.flatnonzero(y01 == 0)
rng = np.random.default_rng(42)
rng.shuffle(positive)
rng.shuffle(negative)
n_val_pos = int(0.2 * len(positive))
n_val_neg = int(0.2 * len(negative))
val_idx = np.r_[positive[:n_val_pos], negative[:n_val_neg]]
fit_idx = np.r_[positive[n_val_pos:], negative[n_val_neg:]]
rng.shuffle(val_idx)
rng.shuffle(fit_idx)

w, loss = logistic_regression(y01[fit_idx], tx_train[fit_idx], np.zeros(tx_train.shape[1]), 20, 0.05)
scores = tx_train[val_idx] @ w
pred = np.where(scores >= 0, 1, -1)
truth = np.where(y01[val_idx] == 1, 1, -1)
accuracy = np.mean(pred == truth)
balanced_accuracy = 0.5 * (np.mean(pred[truth == -1] == -1) + np.mean(pred[truth == 1] == 1))
print("validation loss:", loss)
print("validation accuracy:", accuracy)
print("validation balanced accuracy:", balanced_accuracy)

Current iteration=0, loss=0.6931471805599453
validation loss: 0.5383153494404749
validation accuracy: 0.8495893458485074
validation balanced accuracy: 0.698464801242155


## 3. Fit all labelled data and create the submission

Run this only after choosing hyperparameters using validation or cross-validation.

In [10]:
w_final, final_loss = logistic_regression(y01, tx_train, np.zeros(tx_train.shape[1]), 20, 0.05)
test_scores = tx_test @ w_final
y_pred = np.where(test_scores >= 0, 1, -1)
create_csv_submission(ids_test, y_pred, "submission_logistic_baseline.csv")
print("saved submission_logistic_baseline.csv")
print("prediction counts:", np.unique(y_pred, return_counts=True))

Current iteration=0, loss=0.6931471805599453
saved submission_logistic_baseline.csv
prediction counts: (array([-1,  1]), array([92953, 16426]))
